# Module 3: Light-Matter Interaction & Cavity QED
## Notebook 3.4: The Quantum Rabi Model & Dressed States

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qutip/computational-quantum-optics/blob/main/module_3_light_matter_cavity_qed/04_quantum_rabi_and_dressed_states.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/qutip/computational-quantum-optics)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-brightgreen?logo=python)](https://www.python.org/)

---

### Computational Quantum Optics Curriculum
* **Module 0:** Mathematical & Quantum Foundations
* **Module 1:** Single-Mode Quantum States of Light
* **Module 2:** Phase Space Quasiprobability Distributions
* **Module 3:** **Light-Matter Interaction & Cavity QED**
  * **3.1** Two-Level Atoms & Optical Bloch Equations
  * **3.2** Rabi Oscillations, Detuning & Pulse Control
  * **3.3** The Jaynes-Cummings Hamiltonian & RWA
  * **3.4 The Quantum Rabi Model & Dressed States** $\leftarrow$ *(You are here)*
  * **3.5** Quantum Collapse and Revival of Atomic Inversion
  * **3.6** Resonance Fluorescence & The Mollow Triplet
  * **3.7** Electromagnetically Induced Transparency (EIT) & Dark States

---

### Learning Objectives
1. **Coupling Regimes of Cavity QED:** Define the boundaries between Weak ($g \ll \kappa, \gamma$), Strong ($g \gg \kappa, \gamma$), Ultra-Strong ($0.1 \le g/\omega_c \le 1.0$), and Deep-Strong ($g/\omega_c > 1.0$) coupling.
2. **Breakdown of the Rotating Wave Approximation:** Formulate the full Quantum Rabi Model (QRM) including counter-rotating terms $\hat{a}\hat{\sigma}_- + \hat{a}^\dagger\hat{\sigma}_+$ and identify the breakdown of continuous excitation number conservation.
3. **$\mathbb{Z}_2$ Parity Symmetry:** Prove that the discrete parity operator $\hat{\Pi} = -\hat{\sigma}_z \exp(i\pi\hat{a}^\dagger\hat{a})$ commutes with $\hat{H}_{\text{QRM}}$, sorting eigenstates into decoupled even and odd parity chains.
4. **Virtual Photons & Dressed Ground State:** Calculate the non-zero ground-state virtual photon population $\langle \hat{n} \rangle_{\text{GS}} > 0$ and reconstruct its phase-space Wigner distribution.
5. **Energy Spectrum & Avoided Crossings:** Contrast the Jaynes-Cummings ladder against the non-integrable/analytically solvable QRM spectrum, observing parity-protected level crossings and avoided crossings.


In [ ]:
# Environment Setup: Silent Colab detection & high-DPI publication styling
import os
import sys

# Configure headless Agg backend to ensure bulletproof plotting across all environments
if "MPLBACKEND" not in os.environ:
    os.environ["MPLBACKEND"] = "Agg"

import math
import numpy as np
import scipy.linalg as la
from scipy.special import genlaguerre
import matplotlib.pyplot as plt

# Configure publication matplotlib styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "lines.linewidth": 2.0,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

try:
    import qutip as qt
    HAS_QUTIP = True
    print("QuTiP successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


---

## 1. Coupling Regimes: From Atomic Physics to Circuit QED

In standard cavity QED with Rydberg atoms or optical mirrors, the single-photon coupling strength $g$ is typically a small fraction of the optical frequency ($g/\omega_c \sim 10^{-7} - 10^{-5}$). In this regime, the Rotating Wave Approximation (RWA) is extraordinarily accurate.

However, modern solid-state quantum platforms—particularly **circuit QED** with superconducting transmon/flux qubits coupled to microwave transmission line resonators, polaritonic microcavities, and terahertz Landau polaritons—can achieve unprecedented coupling ratios:

| Regime | Parameter Criterion | Defining Physics |
| :--- | :--- | :--- |
| **Weak Coupling** | $g < \kappa, \gamma$ | Irreversible Purcell decay; perturbative dynamics |
| **Strong Coupling** | $g \gg \kappa, \gamma$ but $g/\omega_c < 0.1$ | Vacuum Rabi oscillations; Jaynes-Cummings ladder valid |
| **Ultra-Strong Coupling (USC)** | $0.1 \le g/\omega_c \le 1.0$ | RWA breaks down; counter-rotating terms active; virtual ground photons |
| **Deep-Strong Coupling (DSC)** | $g/\omega_c > 1.0$ | Coupling exceeds photon energy; ground state is a Schrödinger cat state |

---

## 2. The Full Quantum Rabi Model (QRM)

Without making the Rotating Wave Approximation, the full dipole interaction retains all terms:
$$\hat{H}_{\text{QRM}} = \hbar \omega_c \hat{a}^\dagger \hat{a} + \frac{\hbar \omega_a}{2} \hat{\sigma}_z + \hbar g (\hat{a} + \hat{a}^\dagger)(\hat{\sigma}_+ + \hat{\sigma}_-)$$

Using $\hat{\sigma}_x = \hat{\sigma}_+ + \hat{\sigma}_-$ and the quadrature field operator $\hat{x} = \frac{1}{\sqrt{2}}(\hat{a} + \hat{a}^\dagger)$, the Hamiltonian can be written compactly as:
$$\hat{H}_{\text{QRM}} = \hbar \omega_c \hat{a}^\dagger \hat{a} + \frac{\hbar \omega_a}{2} \hat{\sigma}_z + \hbar g \hat{\sigma}_x (\hat{a} + \hat{a}^\dagger)$$

### Breakdown of Continuous $U(1)$ Symmetry
In the Jaynes-Cummings model, the continuous phase rotation $\hat{a} \to \hat{a} e^{-i\phi}, \hat{\sigma}_- \to \hat{\sigma}_- e^{-i\phi}$ leaves the Hamiltonian invariant, ensuring conservation of total excitations $[\hat{H}_{\text{JC}}, \hat{N}] = 0$.

In the Quantum Rabi Model:
$$[\hat{H}_{\text{QRM}}, \hat{N}] = \hbar g (\hat{a}^\dagger \hat{\sigma}_+ - \hat{a} \hat{\sigma}_-) \ne 0$$
The counter-rotating terms $\hat{a}^\dagger \hat{\sigma}_+$ (simultaneous excitation + photon creation) and $\hat{a} \hat{\sigma}_-$ (simultaneous de-excitation + photon destruction) **couple subspaces differing by $\pm 2$ excitations**!

---

## 3. Discrete $\mathbb{Z}_2$ Parity Symmetry & The Ground State

Although total excitation number $\hat{N}$ is not conserved, the **excitation parity** is strictly conserved!
Define the parity operator:
$$\hat{\Pi} = -\hat{\sigma}_z \exp\left( i \pi \hat{a}^\dagger \hat{a} \right) = \exp\left( i \pi \hat{N} \right)$$
Because $\hat{\Pi}$ eigenvalues are $+1$ (even parity) and $-1$ (odd parity), and:
$$[\hat{H}_{\text{QRM}}, \hat{\Pi}] = 0$$
the entire infinite-dimensional Hilbert space splits into two uncoupled parity sectors:
* **Even Parity Sector ($\Pi = +1$):** Spanned by $\{|g, 2k+1\rangle, |e, 2k\rangle\}$.
* **Odd Parity Sector ($\Pi = -1$):** Spanned by $\{|g, 2k\rangle, |e, 2k+1\rangle\}$.

States of the same parity exhibit **avoided crossings** when system parameters vary, while states of opposite parity can cross freely without interaction!

### The Non-Trivial Ground State & Virtual Photons
In Jaynes-Cummings, the ground state is always $|g, 0\rangle$ with zero photons.
In the Quantum Rabi Model, counter-rotating terms mix $|g, 0\rangle$ with $|e, 1\rangle, |g, 2\rangle, \dots$.
To second-order perturbation theory ($g \ll \omega_c + \omega_a$):
$$|\psi_{\text{GS}}\rangle \approx |g, 0\rangle - \frac{g}{\omega_a + \omega_c} |e, 1\rangle$$
The ground state acquires a **finite virtual photon population**:
$$\langle \hat{n} \rangle_{\text{GS}} = \langle \psi_{\text{GS}} | \hat{a}^\dagger \hat{a} | \psi_{\text{GS}} \rangle \approx \left( \frac{g}{\omega_a + \omega_c} \right)^2 > 0$$
In the Deep Strong Coupling limit ($g/\omega_c \gg 1$), the ground state converges to an entangled **Schrödinger kitten / cat state**:
$$|\psi_{\text{GS}}\rangle \approx \frac{1}{\sqrt{2}} \left( |-\alpha\rangle |-\rangle_x + |\alpha\rangle |+\rangle_x \right), \quad \alpha = \frac{g}{\omega_c}$$


In [ ]:
# Implementation: Full Quantum Rabi Model (QRM) vs. Jaynes-Cummings (JC) Solver

class QuantumRabiSimulator:
    """
    Simulates the Quantum Rabi Model and Jaynes-Cummings model in truncated Fock space:
    H_QRM = hbar*omega_c*a^\dagger a + hbar*omega_a/2 * sigma_z + hbar*g * sigma_x * (a + a^\dagger)
    """
    def __init__(self, omega_c=1.0, omega_a=1.0, g=0.1, n_cutoff=30):
        self.wc = float(omega_c)
        self.wa = float(omega_a)
        self.g = float(g)
        self.n_cut = int(n_cutoff)
        self.dim = 2 * (self.n_cut + 1)
        
        self._build_operators()
        
    def _idx(self, state, n):
        """Matrix index for atomic state 'e' or 'g' with Fock state n."""
        return n if state == 'e' else (self.n_cut + 1) + n
        
    def _build_operators(self):
        """Constructs QRM Hamiltonian, JC Hamiltonian, and Parity operator."""
        H_qrm = np.zeros((self.dim, self.dim), dtype=complex)
        H_jc = np.zeros((self.dim, self.dim), dtype=complex)
        parity = np.zeros((self.dim, self.dim), dtype=complex)
        n_op = np.zeros((self.dim, self.dim), dtype=complex)
        
        # 1. Bare Diagonal terms: hbar*omega_c*n + hbar*omega_a/2 * sigma_z
        for n in range(self.n_cut + 1):
            ie = self._idx('e', n)
            ig = self._idx('g', n)
            
            ee_diag = self.wc * n + 0.5 * self.wa
            gg_diag = self.wc * n - 0.5 * self.wa
            
            H_qrm[ie, ie] = ee_diag
            H_qrm[ig, ig] = gg_diag
            H_jc[ie, ie] = ee_diag
            H_jc[ig, ig] = gg_diag
            
            # Photon number operator
            n_op[ie, ie] = n
            n_op[ig, ig] = n
            
            # Parity operator: Pi = -sigma_z * (-1)^n
            # For |e, n>: - (+1) * (-1)^n = -(-1)^n = (-1)^(n+1)
            # For |g, n>: - (-1) * (-1)^n = +(-1)^n
            parity[ie, ie] = - ((-1)**n)
            parity[ig, ig] = + ((-1)**n)
            
        # 2. Off-diagonal interaction terms
        for n in range(self.n_cut):
            ie = self._idx('e', n)
            ig_next = self._idx('g', n + 1)
            
            ie_next = self._idx('e', n + 1)
            ig = self._idx('g', n)
            
            sqrt_n1 = np.sqrt(n + 1)
            g_eff = self.g * sqrt_n1
            
            # Rotating wave terms: |g, n+1> <-> |e, n>
            H_jc[ie, ig_next] += g_eff
            H_jc[ig_next, ie] += g_eff
            
            H_qrm[ie, ig_next] += g_eff
            H_qrm[ig_next, ie] += g_eff
            
            # Counter-rotating terms: |e, n+1> <-> |g, n> (Only in QRM!)
            H_qrm[ie_next, ig] += g_eff
            H_qrm[ig, ie_next] += g_eff
            
        self.H_qrm = H_qrm
        self.H_jc = H_jc
        self.parity = parity
        self.n_op = n_op
        
    def diagonalize(self, use_rwa=False):
        """Diagonalizes Hamiltonian and sorts eigenstates by energy."""
        H = self.H_jc if use_rwa else self.H_qrm
        evals, evecs = la.eigh(H)
        return evals, evecs
        
    def ground_state_properties(self, use_rwa=False):
        """Calculates ground state energy, virtual photon number <n>, and parity."""
        evals, evecs = self.diagonalize(use_rwa=use_rwa)
        gs = evecs[:, 0]
        e_gs = evals[0]
        n_photons = np.real(np.vdot(gs, self.n_op @ gs))
        p_val = np.real(np.vdot(gs, self.parity @ gs))
        return e_gs, n_photons, p_val, gs

# Verification: Confirm Parity Commutation [H_QRM, Pi] = 0
test_sim = QuantumRabiSimulator(omega_c=1.0, omega_a=1.0, g=0.5, n_cutoff=25)
comm = test_sim.H_qrm @ test_sim.parity - test_sim.parity @ test_sim.H_qrm
comm_norm = la.norm(comm)

print("--- Quantum Rabi Model Symmetry Verification ---")
print(f"Norm of Parity Commutator ||[H_QRM, Pi]||: {comm_norm:.3e}")
assert comm_norm < 1e-12, "Parity symmetry commutation check failed!"
print("Verification Passed: Discrete Z2 Parity symmetry is strictly conserved in the Quantum Rabi Model!")


In [ ]:
# Helper: Compute Reduced Cavity Wigner Function from Composite State Vector

def compute_cavity_wigner(psi_composite, n_cut, xvec, pvec):
    """
    Computes the 2D Wigner function W(x, p) of the cavity field
    by partial-tracing the atomic degree of freedom.
    """
    # Extract atomic components: psi_e(n) and psi_g(n)
    psi_e = psi_composite[:n_cut + 1]
    psi_g = psi_composite[n_cut + 1:2 * (n_cut + 1)]
    
    # Reduced cavity density matrix: rho_field = |psi_e><psi_e| + |psi_g><psi_g|
    rho_field = np.outer(psi_e, psi_e.conj()) + np.outer(psi_g, psi_g.conj())
    
    # Wigner computation via Fock matrix elements:
    # W(alpha) = (1/pi) * sum_{m, n} rho_{m, n} W_{m, n}(alpha)
    X, P = np.meshgrid(xvec, pvec)
    alpha = (X + 1j * P) / np.sqrt(2.0)
    alpha_sq = np.abs(alpha)**2
    
    W = np.zeros_like(X, dtype=float)
    
    # Compute using diagonal and significant off-diagonal elements
    for m in range(min(n_cut + 1, 15)):
        for n in range(m, min(n_cut + 1, 15)):
            val = rho_field[m, n]
            if abs(val) < 1e-6:
                continue
            # Matrix element W_mn
            L = genlaguerre(m, n - m)(2.0 * alpha_sq)
            phase = (2.0 * alpha)**(n - m) * np.exp(-1j * (n - m) * np.angle(alpha))
            w_mn = ((-1)**m / np.pi) * np.sqrt(float(math.factorial(m)) / float(math.factorial(n))) * \
                   (np.sqrt(2.0) * np.conj(alpha))**(n - m) * np.exp(-alpha_sq) * L
                   
            if m == n:
                W += np.real(val * w_mn)
            else:
                W += 2.0 * np.real(val * w_mn)
                
    return W


In [ ]:
# Publication-Quality 4-Panel Visualization: QRM vs JC Spectrum, Virtual Photons & Ground State Wigner

fig = plt.figure(figsize=(16, 12))

# =============================================================================
# Panel 1: Energy Spectrum: JC Ladder vs. QRM across Coupling Regimes
# =============================================================================
ax1 = fig.add_subplot(2, 2, 1)

g_scan = np.linspace(0.0, 1.2, 70)
n_levels_plot = 8
qrm_energies = []
jc_energies = []

for g_val in g_scan:
    sim_scan = QuantumRabiSimulator(omega_c=1.0, omega_a=1.0, g=g_val, n_cutoff=25)
    ev_qrm, _ = sim_scan.diagonalize(use_rwa=False)
    ev_jc, _ = sim_scan.diagonalize(use_rwa=True)
    qrm_energies.append(ev_qrm[:n_levels_plot])
    jc_energies.append(ev_jc[:n_levels_plot])

qrm_energies = np.array(qrm_energies)
jc_energies = np.array(jc_energies)

for k in range(n_levels_plot):
    label_qrm = "QRM Spectrum" if k == 0 else ""
    label_jc = "Jaynes-Cummings (RWA)" if k == 0 else ""
    ax1.plot(g_scan, qrm_energies[:, k], 'r-', lw=1.8, label=label_qrm)
    ax1.plot(g_scan, jc_energies[:, k], 'b--', lw=1.2, alpha=0.7, label=label_jc)

# Demarcate Coupling Regimes
ax1.axvline(0.1, color='gray', linestyle=':', lw=1.5)
ax1.axvline(1.0, color='purple', linestyle=':', lw=1.5)
ax1.text(0.02, 3.2, r"Strong" + "\n" + r"($g \ll \omega_c$)", fontsize=9, color='navy')
ax1.text(0.35, 3.2, r"Ultra-Strong (USC)" + "\n" + r"($0.1 \leq g/\omega_c \leq 1.0$)", fontsize=9, color='darkred')
ax1.text(1.02, 3.2, "Deep-Strong\n(DSC)", fontsize=9, color='purple')

ax1.set_xlabel(r"Normalized Coupling Strength $g / \omega_c$")
ax1.set_ylabel(r"Eigenenergies $E / (\hbar\omega_c)$")
ax1.set_title(r"Energy Spectrum: QRM (Solid Red) vs. JC (Dashed Blue)")
ax1.set_ylim(-1.8, 3.8)
ax1.legend(loc="lower left", frameon=True)

# =============================================================================
# Panel 2: Ground State Virtual Photon Population <n>_GS vs. g/omega_c
# =============================================================================
ax2 = fig.add_subplot(2, 2, 2)

n_photons_qrm = []
n_photons_jc = []
n_photons_pert = []

for g_val in g_scan:
    sim_scan = QuantumRabiSimulator(omega_c=1.0, omega_a=1.0, g=g_val, n_cutoff=25)
    _, n_q, _, _ = sim_scan.ground_state_properties(use_rwa=False)
    _, n_j, _, _ = sim_scan.ground_state_properties(use_rwa=True)
    n_photons_qrm.append(n_q)
    n_photons_jc.append(n_j)
    # Perturbation theory: (g / (wa + wc))^2 = (g / 2)^2 = g^2 / 4
    n_photons_pert.append((g_val / 2.0)**2)

ax2.plot(g_scan, n_photons_qrm, 'r-', lw=2.5, label=r"QRM Numerical Ground State $\langle \hat{n} \rangle_{\mathrm{GS}}$")
ax2.plot(g_scan, n_photons_pert, 'k:', lw=2.0, label=r"Perturbative Scaling $(g/2\omega_c)^2$")
ax2.plot(g_scan, n_photons_jc, 'b--', lw=2.0, label=r"Jaynes-Cummings Vacuum $\langle \hat{n} \rangle_{\mathrm{GS}} \equiv 0$")

ax2.set_xlabel(r"Normalized Coupling $g/\omega_c$")
ax2.set_ylabel(r"Virtual Photon Number $\langle \hat{n} \rangle_{\mathrm{GS}}$")
ax2.set_title("Ground State Dressing: Emergence of Virtual Photons")
ax2.set_ylim(-0.02, 0.75)
ax2.legend(loc="upper left", frameon=True)

# =============================================================================
# Panel 3: Parity Sorting of QRM Spectrum (Avoided vs. Real Crossings)
# =============================================================================
ax3 = fig.add_subplot(2, 2, 3)

for g_val in g_scan[::2]:
    sim_p = QuantumRabiSimulator(omega_c=1.0, omega_a=1.0, g=g_val, n_cutoff=20)
    evals, evecs = sim_p.diagonalize(use_rwa=False)
    for k in range(6):
        # Calculate parity for eigenstate k
        par = np.real(np.vdot(evecs[:, k], sim_p.parity @ evecs[:, k]))
        c = 'blue' if par > 0 else 'red'
        ax3.scatter(g_val, evals[k], color=c, s=12, alpha=0.7)

ax3.plot([], [], 'bo', markersize=6, label=r"Even Parity $\Pi = +1$")
ax3.plot([], [], 'ro', markersize=6, label=r"Odd Parity $\Pi = -1$")
ax3.set_xlabel(r"Coupling Strength $g/\omega_c$")
ax3.set_ylabel(r"Eigenenergies $E / (\hbar\omega_c)$")
ax3.set_title(r"$\mathbb{Z}_2$ Parity Symmetry: Level Crossings & Avoided Crossings")
ax3.set_ylim(-1.8, 3.2)
ax3.legend(loc="upper right", frameon=True)

# =============================================================================
# Panel 4: Cavity Wigner Function in Deep-Strong Coupling Ground State (g = 1.0)
# =============================================================================
ax4 = fig.add_subplot(2, 2, 4)

sim_dsc = QuantumRabiSimulator(omega_c=1.0, omega_a=1.0, g=1.0, n_cutoff=25)
_, _, _, gs_dsc = sim_dsc.ground_state_properties(use_rwa=False)

x_grid = np.linspace(-3.5, 3.5, 120)
p_grid = np.linspace(-3.5, 3.5, 120)
W_gs = compute_cavity_wigner(gs_dsc, sim_dsc.n_cut, x_grid, p_grid)

im4 = ax4.contourf(x_grid, p_grid, W_gs, levels=50, cmap='coolwarm')
ax4.set_xlabel(r"Cavity Quadrature $x$")
ax4.set_ylabel(r"Cavity Quadrature $p$")
ax4.set_title(r"Cavity Ground State Wigner $W(x, p)$ at $g/\omega_c = 1.0$")
plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


> **Physical Insight: Can We Extract Energy from Virtual Ground-State Photons?**
>
> In the ultra-strong and deep-strong coupling regimes, the quantum ground state contains a significant population of virtual photons ($\langle \hat{n} \rangle_{\text{GS}} > 0.5$). A tempting question arises: *Can we place a photodetector next to the cavity and harvest free energy from these ground-state photons?*
>
> The answer is **strictly no**. The state $|\psi_{\text{GS}}\rangle$ is the stationary eigenstate with the **lowest possible energy eigenvalue** of the combined system. Extracting energy or emitting a real traveling photon would require transitioning to a state with even lower energy, which does not exist! The virtual photons are quantum fluctuations bound inextricably to the atomic dipole.
>
> **However**, if one modulates the coupling strength dynamically on ultrafast timescales ($g \to g(t)$ with frequency $\sim 2\omega_c$), this time-dependent perturbation injects energy into the system, releasing the bound virtual photons into real, detectable, propagating photon pairs! This is the physical mechanism of the **Dynamical Casimir Effect** and **parametric vacuum amplification**, experimentally observed in superconducting circuits by Wilson et al. (Nature 2011).


In [ ]:
# Interactive Exploration: Ground State Photon Distribution vs. Coupling Strength

def plot_interactive_qrm(g_coupling=0.5):
    """
    Simulates the Quantum Rabi Model ground state and visualizes the
    photon number probability distribution P(n) in the dressed vacuum.
    """
    sim = QuantumRabiSimulator(omega_c=1.0, omega_a=1.0, g=g_coupling, n_cutoff=25)
    e_gs, n_virt, parity_val, gs_vec = sim.ground_state_properties(use_rwa=False)
    
    # Calculate cavity photon number distribution P(n) = |psi_e(n)|^2 + |psi_g(n)|^2
    psi_e = gs_vec[:sim.n_cut + 1]
    psi_g = gs_vec[sim.n_cut + 1:]
    p_n = np.abs(psi_e)**2 + np.abs(psi_g)**2
    
    fig, (ax_bar, ax_stat) = plt.subplots(1, 2, figsize=(14, 4.5))
    
    # Photon distribution
    n_display = 12
    bars = ax_bar.bar(range(n_display), p_n[:n_display], color='steelblue', edgecolor='navy', alpha=0.8)
    ax_bar.set_xlabel("Photon Fock State $n$")
    ax_bar.set_ylabel("Probability $P(n)$")
    ax_bar.set_title(rf"Dressed Ground State Photon Distribution ($g/\omega_c = {g_coupling:.2f}$)")
    ax_bar.set_xticks(range(n_display))
    ax_bar.set_ylim(0.0, 1.05)
    
    # Text summary of metrics
    ax_stat.axis('off')
    summary_text = (
        f"QUANTUM RABI GROUND STATE METRICS\n"
        f"-----------------------------------------\n"
        f"Coupling Ratio g / omega_c:        {g_coupling:.3f}\n"
        f"Ground State Energy E_GS:          {e_gs:.4f} hbar*omega_c\n"
        f"Virtual Photon Number <n>:         {n_virt:.4f}\n"
        f"Excitation Parity <Pi>:            {parity_val:+.4f} (Even Parity)\n"
        f"Vacuum Component P(0):             {p_n[0]*100:.2f}%\n"
        f"Two-Photon Component P(2):         {p_n[2]*100:.2f}%\n"
        f"\nPhysical Regime: "
        f"{'Strong (RWA Valid)' if g_coupling < 0.1 else 'Ultra-Strong Coupling (USC)' if g_coupling <= 1.0 else 'Deep-Strong Coupling (DSC)'}"
    )
    ax_stat.text(0.05, 0.5, summary_text, fontsize=11, family='monospace', verticalalignment='center',
                 bbox=dict(boxstyle='round,pad=0.8', facecolor='linen', edgecolor='gray', alpha=0.5))
                 
    plt.tight_layout()
    plt.show()

if HAS_WIDGETS:
    widgets.interact(
        plot_interactive_qrm,
        g_coupling=widgets.FloatSlider(value=0.5, min=0.0, max=1.2, step=0.05, description=r"$g/\omega_c$:")
    )
else:
    print("Static exploration demonstration (default parameter: g / omega_c = 0.5):")
    plot_interactive_qrm(g_coupling=0.5)


---

## 4. Self-Check Exercises & Guided Challenge

### Challenge 1: Second-Order Ground-State Energy Shift
Using second-order non-degenerate perturbation theory in the atom-cavity coupling $\hat{V} = \hbar g (\hat{a} + \hat{a}^\dagger)(\hat{\sigma}_+ + \hat{\sigma}_-)$, derive the ground state energy shift for an on-resonance system ($\omega_a = \omega_c = \omega$).

<details>
<summary><b>Solution & Derivation</b> (Click to expand)</summary>

The unperturbed ground state is $|g, 0\rangle$ with zero-order energy $E_0^{(0)} = -\frac{\hbar\omega}{2}$.
The perturbation operator is:
$$\hat{V} = \hbar g \left( \hat{a}\hat{\sigma}_+ + \hat{a}^\dagger\hat{\sigma}_- + \hat{a}^\dagger\hat{\sigma}_+ + \hat{a}\hat{\sigma}_- \right)$$
First-order correction vanishes because $\hat{V}$ has no diagonal elements:
$$E_0^{(1)} = \langle g, 0|\hat{V}|g, 0\rangle = 0$$

For the second-order correction:
$$E_0^{(2)} = \sum_{k \ne (g,0)} \frac{|\langle k|\hat{V}|g, 0\rangle|^2}{E_{(g,0)}^{(0)} - E_k^{(0)}}$$
Acting with $\hat{V}$ on $|g, 0\rangle$:
* $\hat{a}\hat{\sigma}_+ |g, 0\rangle = 0$
* $\hat{a}^\dagger\hat{\sigma}_- |g, 0\rangle = 0$
* $\hat{a}\hat{\sigma}_- |g, 0\rangle = 0$
* $\hat{a}^\dagger\hat{\sigma}_+ |g, 0\rangle = |e, 1\rangle$ (counter-rotating term!)

Thus, **only the state $|e, 1\rangle$ contributes**:
$$\langle e, 1|\hat{V}|g, 0\rangle = \hbar g$$
The energy denominator is:
$$E_{(g,0)}^{(0)} - E_{(e,1)}^{(0)} = \left( -\frac{\hbar\omega_a}{2} \right) - \left( \hbar\omega_c + \frac{\hbar\omega_a}{2} \right) = -(\hbar\omega_a + \hbar\omega_c)$$
Therefore, the second-order ground-state energy shift is:
$$E_0^{(2)} = -\frac{(\hbar g)^2}{\hbar(\omega_a + \omega_c)} = -\frac{\hbar g^2}{\omega_a + \omega_c}$$
On resonance ($\omega_a = \omega_c = \omega$):
$$E_{\text{GS}} \approx -\frac{\hbar\omega}{2} - \frac{\hbar g^2}{2\omega}$$
In Jaynes-Cummings, the counter-rotating term is omitted, so $E_0^{(2)} \equiv 0$ identically!
</details>

---

### Challenge 2: Numerical Verification of Perturbative Energy Shift
For $\omega_c = \omega_a = 1.0$ and moderate coupling $g = 0.2\omega_c$, compute the numerical ground state energy from matrix diagonalization. Compare with the unperturbed value $-0.5$ and the second-order perturbative prediction:
$$E_{\text{pred}} = -0.5 - \frac{0.2^2}{2} = -0.5 - 0.02 = -0.520000$$


In [ ]:
# Numerical Verification of Challenge 2: Second-Order QRM Energy Shift

wc_val = 1.0
wa_val = 1.0
g_test = 0.2

sim_pert = QuantumRabiSimulator(omega_c=wc_val, omega_a=wa_val, g=g_test, n_cutoff=25)
e_gs_num, _, _, _ = sim_pert.ground_state_properties(use_rwa=False)
e_gs_jc, _, _, _ = sim_pert.ground_state_properties(use_rwa=True)

# Theoretical predictions
e_unperturbed = -0.5 * wa_val
e_pert_second_order = e_unperturbed - (g_test**2 / (wa_val + wc_val))

print(f"--- Quantum Rabi Ground State Energy Benchmark (g = {g_test} omega_c) ---")
print(f"Unperturbed Energy:            {e_unperturbed:.6f}")
print(f"Jaynes-Cummings Energy:        {e_gs_jc:.6f} (Strictly unperturbed)")
print(f"2nd-Order Perturbative Energy: {e_pert_second_order:.6f}")
print(f"Full Numerical QRM Energy:     {e_gs_num:.6f}")
print(f"Difference (Num - 2nd Order):  {abs(e_gs_num - e_pert_second_order):.4e}")

assert abs(e_gs_jc - e_unperturbed) < 1e-12, "Jaynes-Cummings vacuum check failed!"
assert abs(e_gs_num - e_pert_second_order) < 0.001, "Second-order perturbation check failed!"
print("Verification Passed: Numerical QRM ground state energy matches 2nd-order perturbation theory to < 0.1% accuracy!")


---

## 5. Summary & Key Formulas

| Concept | Mathematical Expression | Physical Interpretation |
| :--- | :--- | :--- |
| **Full Quantum Rabi Model** | $\hat{H}_{\text{QRM}} = \hbar\omega_c\hat{a}^\dagger\hat{a} + \frac{\hbar\omega_a}{2}\hat{\sigma}_z + \hbar g\hat{\sigma}_x(\hat{a}+\hat{a}^\dagger)$ | Exact atom-cavity interaction without RWA |
| **Parity Operator** | $\hat{\Pi} = -\hat{\sigma}_z \exp(i\pi\hat{a}^\dagger\hat{a})$ ($[\hat{H}, \hat{\Pi}] = 0$) | $\mathbb{Z}_2$ discrete parity symmetry; decouples even and odd chains |
| **Virtual Photon Number** | $\langle\hat{n}\rangle_{\text{GS}} \approx \left(\frac{g}{\omega_a+\omega_c}\right)^2$ | Non-zero ground state photons generated by counter-rotating terms |
| **USC Threshold** | $g / \omega_c \ge 0.1$ | Counter-rotating terms cannot be neglected |
| **DSC Regime** | $g / \omega_c > 1.0$ | Ground state approaches an entangled Schrödinger cat state |

---

### Further Reading & Foundational Literature
* **I. I. Rabi**, *On the Process of Space Quantization*, Phys. Rev. 49, 324 (1936).
* **D. Braak**, *Integrability of the Quantum Rabi Model*, Phys. Rev. Lett. 107, 100401 (2011).
* **A. Frisk Kockum, A. Miranowicz, S. De Liberato, S. Savasta, & F. Nori**, *Ultrastrong Coupling between Light and Matter*, Nature Reviews Physics 1, 19–40 (2019).

---

### Next Notebook in Module 3
In **Notebook 3.5: Quantum Collapse and Revival of Atomic Inversion**, we explore the rich multi-photon dynamics when an atom interacts with a **coherent state** $|\alpha\rangle$:
* Dephasing of incommensurate Rabi frequencies $\Omega_n = 2g\sqrt{n+1}$ producing the **collapse** of Rabi oscillations.
* Rephasing of quantum amplitudes leading to macroscopic **revivals** at time $t_{\text{rev}} = 2\pi\sqrt{\bar{n}}/g$.
* Fractional revivals at $t_{\text{rev}}/2$ and the transient generation of **Schrödinger cat states** in the cavity field!
